Installing required libraries

In [ ]:
!pip install -q transformers datasets scikit-learn sentence-transformers pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 101.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 60.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 65.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 43.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 36.8 MB/s eta 0:00:00


Uploading and preparing the dataset

In [ ]:
from google.colab import files
import pandas as pd

uploaded = files.upload()
df = pd.read_csv(list(uploaded.keys())[0])
df['text'] = df[['title', 'abstract', 'summary_comments']].fillna('').agg(' '.join, axis=1)
df['label'] = df['accepted'].astype(int)


Saving preprocessed_dataset.csv to preprocessed_dataset.csv


Training bert classification model

In [ ]:
from datasets import Dataset
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

features = ['text', 'novelty_score', 'quality_score', 'relevance_score', 'review_sentiment', 'label']
df = df[features]
train_df, val_df = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=42)

train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
val_ds = Dataset.from_pandas(val_df.reset_index(drop=True))

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
def tokenize(example): return tokenizer(example["text"], truncation=True, padding="max_length", max_length=512)
train_ds = train_ds.map(tokenize, batched=True)
val_ds = val_ds.map(tokenize, batched=True)

model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    return {"accuracy": accuracy_score(labels, preds), "f1": f1_score(labels, preds)}

training_args = TrainingArguments(
    output_dir="./bert_acceptance",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_dir="./logs"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()
trainer.save_model("latest_model")
tokenizer.save_pretrained("latest_model")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

Map:   0%|          | 0/64 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-3-268466573.py:41: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

Loading BERT and setting up pipeline

In [ ]:
from transformers import TextClassificationPipeline, AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained("latest_model")
model = AutoModelForSequenceClassification.from_pretrained("latest_model")
clf = TextClassificationPipeline(model=model, tokenizer=tokenizer, return_all_scores=True)


Device set to use cuda:0
/usr/local/lib/python3.11/dist-packages/transformers/pipelines/text_classification.py:106: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


training score regressors and classfiers

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestClassifier

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
text_embeddings = embedding_model.encode(df['text'].tolist(), show_progress_bar=True)

target_scores = ['novelty_score', 'quality_score', 'relevance_score', 'review_sentiment']
X = text_embeddings
y_scores = df[target_scores]
y_label = df['label']

score_predictors = {}
for score in target_scores:
    model_r = Ridge()
    model_r.fit(X, y_scores[score])
    score_predictors[score] = model_r

score_model = RandomForestClassifier(n_estimators=100, random_state=42)
score_model.fit(y_scores, y_label)


Batches:   0%|          | 0/21 [00:00<?, ?it/s]

RandomForestClassifier(random_state=42)

Uploading pdf and getting final verdict

In [ ]:
import fitz  # PyMuPDF

uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]

def extract_text(path):
    doc = fitz.open(path)
    return " ".join(page.get_text() for page in doc)

paper_text = extract_text(pdf_path)

paper_embedding = embedding_model.encode([paper_text])[0]
bert_result = clf(paper_text[:512])[0]
bert_label = max(bert_result, key=lambda x: x['score'])['label']
bert_conf = max(bert_result, key=lambda x: x['score'])['score']
bert_verdict = "ACCEPTED" if bert_label == "LABEL_1" else "REJECTED"

predicted_scores = {
    score: round(score_predictors[score].predict([paper_embedding])[0], 2)
    for score in target_scores
}
score_input = [[predicted_scores[s] for s in target_scores]]
score_verdict = "ACCEPTED" if score_model.predict(score_input)[0] == 1 else "REJECTED"

final_verdict = "ACCEPTED" if [bert_verdict, score_verdict].count("ACCEPTED") >= 1 else "REJECTED"

print("\n--- Scores ---")
for k, v in predicted_scores.items():
    print(f"{k}: {v}")

print("\n--- Hybrid Verdict ---")
print(f"BERT Verdict: {bert_verdict} (Confidence: {bert_conf:.2f})")
print(f"Score-Based Verdict: {score_verdict}")
print(f"Final Verdict: {final_verdict}")
print("\n Feedback Comments ")

for score, value in predicted_scores.items():
    if score == 'novelty_score':
        if value >= 0.8:
            comment = "The paper demonstrates a high level of originality."
        elif value >= 0.6:
            comment = "The work shows reasonable novelty but could be more innovative."
        elif value >= 0.4:
            comment = "The ideas presented are somewhat common and originality could be improved."
        else:
            comment = "The paper lacks novelty and closely resembles existing work."

    elif score == 'quality_score':
        if value >= 0.8:
            comment = "The writing is clear, well-structured, and easy to understand."
        elif value >= 0.6:
            comment = "The paper is generally well-written, with minor improvements suggested."
        elif value >= 0.4:
            comment = "Readability is average with noticeable issues in grammar or flow."
        else:
            comment = "The writing quality is poor and requires significant revision."

    elif score == 'relevance_score':
        if value >= 0.8:
            comment = "The topic is highly relevant to the intended research domain."
        elif value >= 0.6:
            comment = "The topic is generally relevant but could be more aligned with the domain."
        elif value >= 0.4:
            comment = "The relevance is moderate and may not fully align with the domain's focus."
        else:
            comment = "The paper appears to be unrelated to the intended research scope."

    elif score == 'review_sentiment':
        if value >= 0.8:
            comment = "The overall impression is highly positive and supportive."
        elif value >= 0.6:
            comment = "The sentiment is somewhat favorable, though not strongly positive."
        elif value >= 0.4:
            comment = "The review sentiment is neutral and lacks clear support."
        else:
            comment = "The sentiment suggests concern or disapproval from the reviewer."

    print(f"{score.replace('_', ' ').title()}: {comment}")



Saving Generating and Exploiting Large-scale Pseudo Training Data for Zero.pdf to Generating and Exploiting Large-scale Pseudo Training Data for Zero.pdf

--- Automatically Generated Scores ---
novelty_score: 2.819999933242798
quality_score: 3.9700000286102295
relevance_score: 3.7899999618530273
review_sentiment: 0.10999999940395355

--- Hybrid Verdict ---
BERT Verdict: REJECTED (Confidence: 0.56)
Score-Based Verdict: REJECTED
Final Verdict: REJECTED


/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


Testing and proving why the model is BEST

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import numpy as np

# Step 1: Split test set
from sklearn.model_selection import train_test_split
train_df, test_df = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=42)

# Step 2: Evaluate BERT-only
bert_preds = []
for text in test_df['text']:
    result = clf(text[:512])[0]
    pred = np.argmax([x['score'] for x in result])  # 0 or 1
    bert_preds.append(pred)

# Step 3: Evaluate Score-only
score_inputs = test_df[['novelty_score', 'quality_score', 'relevance_score', 'review_sentiment']]
score_preds = score_model.predict(score_inputs)

# Step 4: Evaluate Hybrid model
hybrid_preds = []
for idx, row in test_df.iterrows():
    text = row['text']
    scores = [row['novelty_score'], row['quality_score'], row['relevance_score'], row['review_sentiment']]

    # BERT prediction
    result = clf(text[:512])[0]
    bert_pred = np.argmax([x['score'] for x in result])

    # Score model prediction
    score_pred = score_model.predict([scores])[0]

    # Hybrid rule: if either says ACCEPTED, we accept
    final_pred = 1 if [bert_pred, score_pred].count(1) >= 1 else 0
    hybrid_preds.append(final_pred)

# Step 5: Evaluate all
true_labels = test_df['label'].values

def evaluate(name, preds):
    print(f"\n {name} MODEL")
    print("Accuracy :", round(accuracy_score(true_labels, preds), 3))
    print("F1 Score :", round(f1_score(true_labels, preds), 3))
    print("Precision:", round(precision_score(true_labels, preds), 3))
    print("Recall   :", round(recall_score(true_labels, preds), 3))

# Print results
evaluate("BERT-ONLY", bert_preds)
evaluate("SCORE-ONLY", score_preds)
evaluate("HYBRID", hybrid_preds)


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not h


📊 BERT-ONLY MODEL
Accuracy : 0.511
F1 Score : 0.522
Precision: 0.486
Recall   : 0.562

📊 SCORE-ONLY MODEL
Accuracy : 1.0
F1 Score : 1.0
Precision: 1.0
Recall   : 1.0

📊 HYBRID MODEL
Accuracy : 0.719
F1 Score : 0.771
Precision: 0.627
Recall   : 1.0


/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
